In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from groq import Groq


In [3]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1"#"https://api.groq.com/openai/v1"
)

In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model= "openai/gpt-oss-120b",
        input=prompt
    )
    return response.output_text

In [5]:
llm('hello,whatssapp')

'Hello! How can I help you with WhatsApp today?'

## Simple Retrival
 #### here we are just doing a simple prompt and retrieving the answer, so as you can see any specific answer, can give general answer on which LLM is trained.


In [6]:
question = "I just discovered the course. Can I join now? for llmzoomcamp"
answer = llm(question)
print(answer)

Absolutely! The **LLM Zoomcamp** is designed to be flexible, so you can hop on board at any time. Here’s a quick rundown of what you need to do to get started right away:

---

## 1️⃣ Check the Enrollment Status
- **Open enrollment:** The course is currently open for new participants.  
- **Self‑paced:** All lectures, notebooks, and assignments are available on the course platform, so you can move through the material at your own speed.

> **Tip:** If you see a “Join Now” or “Enroll” button on the course homepage, just click it and follow the sign‑up flow. If you don’t see it, let me know and I can point you to the exact URL.

---

## 2️⃣ Create Your Account
1. **Sign up** with your email (or GitHub/Google) on the course portal.  
2. **Verify** your email address (you’ll get a quick confirmation link).  
3. **Accept the terms** (the usual code‑of‑conduct and data‑privacy notice).

---

## 3️⃣ Set Up the Required Tools (quick checklist)

| Tool | Why you need it | How to install |
|----

## RAG: Crude way 

In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
question1='do you know what date the couse start'

As you can see, we haven't given the answer here because the latest model doesn't require. It's understand to stop the sentence .



In [9]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [10]:
answer = llm(prompt)
print(answer)

Yes—you can still join the LLM Zoomcamp. The course is open for new participants, but if you’d like to earn a certificate you’ll need to submit your final project while the submission window is still open.


In [11]:
answer = llm(prompt)
print(answer)

Yes—you can still join the LLM Zoomcamp. Just start learning and submitting the homework right away. If you’d like to receive a certificate, be sure to submit your final project before the submission deadline (while we’re still accepting projects).


### RAG pieces

- **Search**: retrieves the most relevant documents or chunks from a knowledge source.
- **Prompt**: combines the user question with the retrieved context and instructions.
- **LLM**: generates the final answer using that prompt.

### Flow

1. User asks a question.
2. Search finds relevant context.
3. Prompt packages the question + context.
4. LLM produces the response.

## 1.4
- RAG 

In [12]:
def rag(question):
    search_result=search(question)
    user_promt=build_prompt(question,search_result)
    return llm(user_promt)

- fething the data/prepairing the data

In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
response.raise_for_status()
courses_raw = response.json()

In [14]:
courses_raw[0]

{'course': 'data-engineering-zoomcamp',
 'course_name': 'Data Engineering Zoomcamp',
 'path': '/json/data-engineering-zoomcamp.json',
 'questions_count': 404}

In [15]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1401

In [16]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

## 1.5 search

In [17]:
from minsearch import Index

index = Index(
    text_fields=['section','question','answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [18]:
print(question)

I just discovered the course. Can I join now? for llmzoomcamp


In [19]:
question2="I just discovered the course. Can I join now?"

In [20]:
search_result=index.search(
    question1,
    filter_dict={'course':'llm-zoomcamp'}, 
    num_results=4)

In [21]:
search_result

[{'id': 'e394e6f738',
  'course': 'llm-zoomcamp',
  'section': 'Workshop: Open-Source Data Ingestion (dlt)',
  'question': 'How do I know which tables are in the db?',
  'answer': 'You can use the `db.table_names()` method to list all the tables in the database.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work throu

In [22]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=1
    )

In [23]:
search_results=search(question)

## 1.6 Building a prompt

In [24]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [25]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [26]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [27]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [28]:
u_prompt=build_prompt(question1,search_results)
print(u_prompt)

Question:
do you know what date the couse start

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


## 1.7 llm

In [29]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=u_prompt
)

In [30]:
response

Response(id='gen-1786334438-acjpQcgJUnat1gatR4Fm', created_at=1786334438.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='openai/gpt-oss-120b', object='response', output=[ResponseReasoningItem(id='rs_tmp_6677vbav214', summary=[], type='reasoning', content=[Content(text='We need to answer: "do you know what date the couse start". Likely a typo: "course". The context is general course-related Q&A. The answer should be about start date. Probably the answer: The course starts on a specific date, maybe "the course starts on [date]"? But we don\'t have the exact date. The context suggests a generic answer: "The course is self-paced, you can start anytime." Or "The course starts on [date]"? The question: "do you know what date the couse start". Could be a user asking about start date. The answer: "The course is self-paced, you can start anytime, but the submission deadline for certificates is [date]."\n\nThus answer: The course is open and you can start whenever,

In [31]:
response.output[0]

ResponseReasoningItem(id='rs_tmp_6677vbav214', summary=[], type='reasoning', content=[Content(text='We need to answer: "do you know what date the couse start". Likely a typo: "course". The context is general course-related Q&A. The answer should be about start date. Probably the answer: The course starts on a specific date, maybe "the course starts on [date]"? But we don\'t have the exact date. The context suggests a generic answer: "The course is self-paced, you can start anytime." Or "The course starts on [date]"? The question: "do you know what date the couse start". Could be a user asking about start date. The answer: "The course is self-paced, you can start anytime, but the submission deadline for certificates is [date]."\n\nThus answer: The course is open and you can start whenever, but if you want a certificate you need to submit before the deadline. Provide that.', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown')

In [32]:
response.output[0].content[0].text

'We need to answer: "do you know what date the couse start". Likely a typo: "course". The context is general course-related Q&A. The answer should be about start date. Probably the answer: The course starts on a specific date, maybe "the course starts on [date]"? But we don\'t have the exact date. The context suggests a generic answer: "The course is self-paced, you can start anytime." Or "The course starts on [date]"? The question: "do you know what date the couse start". Could be a user asking about start date. The answer: "The course is self-paced, you can start anytime, but the submission deadline for certificates is [date]."\n\nThus answer: The course is open and you can start whenever, but if you want a certificate you need to submit before the deadline. Provide that.'

In [33]:
response.output_text

'The course is self‑paced – you can jump in and start working on it at any time.\u202fIf you’re aiming to earn a certificate, just be sure to submit your final project before the current submission deadline (the date listed on the course’s “Certificates” page).  Otherwise there’s no fixed “start date” you need to wait for.'

In [34]:
response.usage

ResponseUsage(input_tokens=113, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0), output_tokens=261, output_tokens_details=OutputTokensDetails(reasoning_tokens=196), total_tokens=374, cost=4.8551e-05, is_byok=False, cost_details={'upstream_inference_cost': 4.8551e-05, 'upstream_inference_input_cost': 4.181e-06, 'upstream_inference_output_cost': 4.437e-05})

In [35]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=message_history
)

In [36]:
def llm(instructions, user_prompt, model="openai/gpt-oss-120b"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [37]:
def rag(query, model="openai/gpt-oss-120b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [38]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes—you can still join the course. Just keep in mind that if you want a certificate, you’ll need to submit your project before the submission deadline.


In [39]:
rag("How do I get a certificate?")

'You can earn a certificate only by completing the course as part of a **live cohort**.  \nHere’s what you need to do:\n\n1. **Finish a live cohort** – self‑paced study alone isn’t enough; the cohort must be open for submissions.  \n2. **Submit a capstone project** – work on the project in whatever mode you prefer, but the final submission must be made while a live cohort is accepting it.  \n3. **Complete the required peer reviews** – after you submit your project, you must review the required number of fellow students’ projects (the same as the cohort’s peer‑review requirement).  \n\n*Homework assignments are not required for the certificate.*  \n\nIn short, prepare your material and project however you like, then submit the capstone and finish the peer‑review tasks during a live cohort session to receive your certificate.'

## 1.8 RAG Helper